# GPTNeo Model Comparison: MHA vs MQA

Comprehensive comparison of Multi-Head Attention (MHA) vs Multi-Query Attention (MQA) architectures trained on TinyStories dataset.

**Comparison Overview:**
- 📊 Training metrics and performance
- 📝 Generated text quality (side-by-side)
- ⚡ Inference speed and memory usage
- 🧠 Embedding space analysis (learned representations)
- ⚖️ Trade-offs and use case recommendations

**Models:**
- Both trained on 30K TinyStories samples, 6K steps
- Same hyperparameters, only attention mechanism differs
- MHA: ~16.1M params | MQA: ~15.6M params (-3%)

## 1. Setup & Imports

In [ ]:
# Install dependencies
!pip install -q transformers datasets scikit-learn matplotlib plotly seaborn torch tqdm pandas

print("✓ Dependencies installed")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import GPT2Tokenizer
from collections import Counter
import pandas as pd
import json
import time
from tqdm import tqdm
import os

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Setup paths and repository
import os

# Check if we're in Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # Remove existing repository if it exists
    if os.path.exists('PROJECT'):
        !rm -rf PROJECT
        print("✓ Existing repository removed")
    
    # Clone repository
    !git clone https://gitlab.cim.rhul.ac.uk/wmis066/PROJECT.git
    print("✓ Repository cloned")
    
    # Change to project directory
    %cd PROJECT

print("✓ Setup complete")

In [ ]:
# Import project modules
import sys
import importlib

if IN_COLAB:
    project_root = '/content/PROJECT'
else:
    project_root = os.path.abspath('../..')

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import transformer modules
from AttentionHeads.mha import transformer as mha_transformer
from AttentionHeads.mqa import transformer as mqa_transformer

print("✓ Project modules imported")

In [ ]:
# ============================================================================
# CONFIGURE MODEL PATH - Update this to point to your trained models
# ============================================================================

# OPTION 1: If running in Colab, specify your Google Drive path or /content path
# MODEL_DIR = '/content/drive/MyDrive/GPTNeo_Models'  # Google Drive
# MODEL_DIR = '/content/models'                       # /content directory

# OPTION 2: If running locally, specify local path
# MODEL_DIR = '/Users/yourusername/path/to/models'   # macOS/Linux
# MODEL_DIR = 'C:/Users/yourusername/path/to/models' # Windows

# Default: Use /content/models (you need to upload/copy models there first)
MODEL_DIR = '/content/models'

print(f"Model directory: {MODEL_DIR}")
print(f"\nExpected files:")
print(f"  • {MODEL_DIR}/best_model_mha.pt")
print(f"  • {MODEL_DIR}/best_model_mqa.pt")

## 2. Model Path Configuration

**Update the `MODEL_DIR` path below to point to your trained models.**

In [ ]:
## 3. Load Models & Checkpoints

## 2. Load Models & Checkpoints

In [ ]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Checkpoint paths
mha_checkpoint_path = os.path.join(MODEL_DIR, 'best_model_mha.pt')
mqa_checkpoint_path = os.path.join(MODEL_DIR, 'best_model_mqa.pt')

# Verify files exist
assert os.path.exists(mha_checkpoint_path), f"MHA checkpoint not found: {mha_checkpoint_path}"
assert os.path.exists(mqa_checkpoint_path), f"MQA checkpoint not found: {mqa_checkpoint_path}"

print("✓ Checkpoint files verified")

In [ ]:
# Load MHA checkpoint
print("Loading MHA checkpoint...")
mha_checkpoint = torch.load(mha_checkpoint_path, map_location=device, weights_only=False)

# Extract MHA info
mha_config = mha_checkpoint['config']
mha_step = mha_checkpoint.get('global_step', 'Unknown')
mha_tokens = mha_checkpoint.get('tokens_seen', 'Unknown')
mha_best_loss = mha_checkpoint.get('best_val_loss', 'Unknown')
mha_metrics = mha_checkpoint.get('metrics', {})
mha_val_ppl = mha_metrics.get('val_ppl', 'Unknown')

print("\n" + "="*70)
print("MHA Model Summary:")
print("="*70)
print(f"  Training step: {mha_step:,}" if isinstance(mha_step, int) else f"  Training step: {mha_step}")
print(f"  Tokens seen: {mha_tokens:,}" if isinstance(mha_tokens, int) else f"  Tokens seen: {mha_tokens}")
print(f"  Best val loss: {mha_best_loss:.4f}" if isinstance(mha_best_loss, float) else f"  Best val loss: {mha_best_loss}")
print(f"  Val perplexity: {mha_val_ppl:.2f}" if isinstance(mha_val_ppl, (int, float)) else f"  Val perplexity: {mha_val_ppl}")
print("="*70)

In [ ]:
# Load MQA checkpoint
print("Loading MQA checkpoint...")
mqa_checkpoint = torch.load(mqa_checkpoint_path, map_location=device, weights_only=False)

# Extract MQA info
mqa_config = mqa_checkpoint['config']
mqa_step = mqa_checkpoint.get('global_step', 'Unknown')
mqa_tokens = mqa_checkpoint.get('tokens_seen', 'Unknown')
mqa_best_loss = mqa_checkpoint.get('best_val_loss', 'Unknown')
mqa_metrics = mqa_checkpoint.get('metrics', {})
mqa_val_ppl = mqa_metrics.get('val_ppl', 'Unknown')

print("\n" + "="*70)
print("MQA Model Summary:")
print("="*70)
print(f"  Training step: {mqa_step:,}" if isinstance(mqa_step, int) else f"  Training step: {mqa_step}")
print(f"  Tokens seen: {mqa_tokens:,}" if isinstance(mqa_tokens, int) else f"  Tokens seen: {mqa_tokens}")
print(f"  Best val loss: {mqa_best_loss:.4f}" if isinstance(mqa_best_loss, float) else f"  Best val loss: {mqa_best_loss}")
print(f"  Val perplexity: {mqa_val_ppl:.2f}" if isinstance(mqa_val_ppl, (int, float)) else f"  Val perplexity: {mqa_val_ppl}")
print("="*70)

In [ ]:
## 4. Architecture Comparison

## 3. Architecture Comparison

In [ ]:
# Count parameters
mha_total_params = mha_model.get_num_params()
mha_non_embed_params = mha_model.get_num_params(non_embedding=True)
mha_embed_params = mha_total_params - mha_non_embed_params

mqa_total_params = mqa_model.get_num_params()
mqa_non_embed_params = mqa_model.get_num_params(non_embedding=True)
mqa_embed_params = mqa_total_params - mqa_non_embed_params

# Calculate differences
total_diff = ((mha_total_params - mqa_total_params) / mha_total_params) * 100
non_embed_diff = ((mha_non_embed_params - mqa_non_embed_params) / mha_non_embed_params) * 100

print("\n" + "="*70)
print("MODEL ARCHITECTURE COMPARISON")
print("="*70)

# Create comparison table
comparison_data = {
    'Metric': [
        'Total Parameters',
        'Embedding Parameters',
        'Non-Embedding Parameters',
        'Hidden Size',
        'Number of Layers',
        'Number of Heads',
        'FFN Size',
        'Max Sequence Length'
    ],
    'MHA': [
        f"{mha_total_params:,}",
        f"{mha_embed_params:,}",
        f"{mha_non_embed_params:,}",
        mha_config['model']['hidden_size'],
        mha_config['model']['num_layers'],
        mha_config['model']['num_heads'],
        mha_config['model']['intermediate_size'],
        mha_config['model']['max_position_embeddings']
    ],
    'MQA': [
        f"{mqa_total_params:,}",
        f"{mqa_embed_params:,}",
        f"{mqa_non_embed_params:,}",
        mqa_config['model']['hidden_size'],
        mqa_config['model']['num_layers'],
        mqa_config['model']['num_heads'],
        mqa_config['model']['intermediate_size'],
        mqa_config['model']['max_position_embeddings']
    ],
    'Difference': [
        f"-{total_diff:.1f}%",
        "Same",
        f"-{non_embed_diff:.1f}%",
        "Same",
        "Same",
        "Same (8 Q, 1 K/V)",
        "Same",
        "Same"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + df_comparison.to_string(index=False))
print("\n" + "="*70)
print(f"\nKey Insight: MQA saves {non_embed_diff:.1f}% non-embedding parameters")
print(f"This comes from using 1 shared K/V head instead of {mha_config['model']['num_heads']} separate K/V heads")
print("="*70)

In [ ]:
## 5. Training Metrics Comparison

## 4. Training Metrics Comparison

In [ ]:
# Training metrics summary
print("\n" + "="*70)
print("TRAINING METRICS COMPARISON")
print("="*70)

metrics_data = {
    'Metric': [
        'Training Steps',
        'Tokens Seen',
        'Best Val Loss',
        'Val Perplexity',
        'Training Samples',
        'Val Samples'
    ],
    'MHA': [
        f"{mha_step:,}" if isinstance(mha_step, int) else str(mha_step),
        f"{mha_tokens:,}" if isinstance(mha_tokens, int) else str(mha_tokens),
        f"{mha_best_loss:.4f}" if isinstance(mha_best_loss, float) else str(mha_best_loss),
        f"{mha_val_ppl:.2f}" if isinstance(mha_val_ppl, (int, float)) else str(mha_val_ppl),
        mha_config['training']['train_samples'],
        mha_config['training']['val_samples']
    ],
    'MQA': [
        f"{mqa_step:,}" if isinstance(mqa_step, int) else str(mqa_step),
        f"{mqa_tokens:,}" if isinstance(mqa_tokens, int) else str(mqa_tokens),
        f"{mqa_best_loss:.4f}" if isinstance(mqa_best_loss, float) else str(mqa_best_loss),
        f"{mqa_val_ppl:.2f}" if isinstance(mqa_val_ppl, (int, float)) else str(mqa_val_ppl),
        mqa_config['training']['train_samples'],
        mqa_config['training']['val_samples']
    ]
}

# Calculate differences
if isinstance(mha_best_loss, float) and isinstance(mqa_best_loss, float):
    loss_diff = ((mqa_best_loss - mha_best_loss) / mha_best_loss) * 100
    metrics_data['Difference'] = [
        'Same',
        'Same',
        f"{'+' if loss_diff > 0 else ''}{loss_diff:.1f}%",
        f"{'+' if mqa_val_ppl > mha_val_ppl else ''}{((mqa_val_ppl - mha_val_ppl) / mha_val_ppl * 100):.1f}%" if isinstance(mqa_val_ppl, (int, float)) and isinstance(mha_val_ppl, (int, float)) else 'N/A',
        'Same',
        'Same'
    ]
else:
    metrics_data['Difference'] = ['N/A'] * 6

df_metrics = pd.DataFrame(metrics_data)
print("\n" + df_metrics.to_string(index=False))
print("\n" + "="*70)

In [ ]:
## 6. Text Generation Quality Comparison

## 5. Text Generation Quality Comparison

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✓ Tokenizer loaded")

In [ ]:
# Define test prompts
test_prompts = [
    "Once upon a time",
    "One day, a little girl",
    "In a big forest",
    "There was a happy dog",
    "A small boy found",
    "The magic toy",
    "When the sun",
    "In the dark cave",
]

# Generation parameters (matching the model's generate() signature)
gen_params = {
    'max_length': 100,
    'temperature': 0.8,
    'top_k': 50,
    'top_p': 0.95
    # Note: The model's generate() always samples, no need for do_sample parameter
}

print(f"Test prompts: {len(test_prompts)}")
print(f"Generation params: {gen_params}")

In [ ]:
# Generate stories from both models
print("\nGenerating stories from both models...\n")

generations = []

for i, prompt in enumerate(test_prompts, 1):
    print(f"Generating {i}/{len(test_prompts)}: '{prompt}'...")
    
    # Tokenize prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate with MHA
    with torch.no_grad():
        mha_output = mha_model.generate(input_ids, **gen_params)
    mha_text = tokenizer.decode(mha_output[0], skip_special_tokens=True)
    
    # Generate with MQA
    with torch.no_grad():
        mqa_output = mqa_model.generate(input_ids, **gen_params)
    mqa_text = tokenizer.decode(mqa_output[0], skip_special_tokens=True)
    
    generations.append({
        'prompt': prompt,
        'mha': mha_text,
        'mqa': mqa_text
    })

print("\n✓ Generation complete")

In [ ]:
# Display side-by-side comparisons
print("\n" + "="*80)
print("TEXT GENERATION COMPARISON")
print("="*80)

for i, gen in enumerate(generations, 1):
    print(f"\n[{i}] Prompt: \"{gen['prompt']}\"")
    print("-" * 80)
    print("\nMHA Generated:")
    print(gen['mha'])
    print("\nMQA Generated:")
    print(gen['mqa'])
    print("-" * 80)

print("\n" + "="*80)

In [ ]:
## 7. Inference Performance Benchmarking

## 6. Inference Performance Benchmarking

In [ ]:
# Benchmark generation speed
print("\nBenchmarking generation speed...\n")

def benchmark_generation(model, num_runs=20, max_length=100):
    """Benchmark generation speed"""
    prompt = "Once upon a time"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Warm-up
    for _ in range(5):
        with torch.no_grad():
            _ = model.generate(input_ids, max_length=50, temperature=1.0)
    
    # Clear cache
    if device == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Benchmark (use temperature=1.0 and top_k=0 for deterministic-like behavior)
    times = []
    for _ in tqdm(range(num_runs), desc="Benchmarking"):
        start = time.time()
        with torch.no_grad():
            output = model.generate(input_ids, max_length=max_length, temperature=1.0, top_k=1)
        
        if device == 'cuda':
            torch.cuda.synchronize()
        
        end = time.time()
        times.append(end - start)
    
    # Calculate tokens generated
    tokens_generated = output.shape[1] - input_ids.shape[1]
    
    return {
        'mean_time': np.mean(times),
        'std_time': np.std(times),
        'median_time': np.median(times),
        'tokens_per_sec': tokens_generated / np.mean(times),
        'tokens_generated': tokens_generated
    }

# Benchmark both models
print("MHA:")
mha_bench = benchmark_generation(mha_model)

print("\nMQA:")
mqa_bench = benchmark_generation(mqa_model)

print("\n" + "="*70)
print("INFERENCE PERFORMANCE")
print("="*70)

perf_data = {
    'Metric': [
        'Mean Time (s)',
        'Std Time (s)',
        'Tokens/Second',
        'Speedup'
    ],
    'MHA': [
        f"{mha_bench['mean_time']:.3f}",
        f"{mha_bench['std_time']:.3f}",
        f"{mha_bench['tokens_per_sec']:.2f}",
        '1.00×'
    ],
    'MQA': [
        f"{mqa_bench['mean_time']:.3f}",
        f"{mqa_bench['std_time']:.3f}",
        f"{mqa_bench['tokens_per_sec']:.2f}",
        f"{mqa_bench['tokens_per_sec'] / mha_bench['tokens_per_sec']:.2f}×"
    ]
}

df_perf = pd.DataFrame(perf_data)
print("\n" + df_perf.to_string(index=False))
print("\n" + "="*70)

speedup = mqa_bench['tokens_per_sec'] / mha_bench['tokens_per_sec']
print(f"\nMQA is {speedup:.2f}× faster than MHA")

In [ ]:
# Visualize performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ['MHA', 'MQA']
colors = ['#3498db', '#e67e22']

# Tokens per second
tokens_per_sec = [mha_bench['tokens_per_sec'], mqa_bench['tokens_per_sec']]
bars1 = axes[0].bar(models, tokens_per_sec, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Tokens / Second', fontsize=11)
axes[0].set_title('Generation Speed', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for bar, val in zip(bars1, tokens_per_sec):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# Generation time
gen_times = [mha_bench['mean_time'], mqa_bench['mean_time']]
bars2 = axes[1].bar(models, gen_times, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Time (seconds)', fontsize=11)
axes[1].set_title('Mean Generation Time', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for bar, val in zip(bars2, gen_times):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}s', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Inference Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('inference_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Performance plots saved")

In [ ]:
## 8. Embedding Space Analysis

## 7. Embedding Space Analysis

In [ ]:
# Extract embeddings from checkpoints
print("Extracting embeddings from models...")

mha_token_emb = mha_checkpoint['model_state_dict']['transformer.token_embedding.weight'].cpu().numpy()
mqa_token_emb = mqa_checkpoint['model_state_dict']['transformer.token_embedding.weight'].cpu().numpy()

print(f"\nMHA token embeddings: {mha_token_emb.shape}")
print(f"MQA token embeddings: {mqa_token_emb.shape}")
print("✓ Embeddings extracted")

In [ ]:
# Compute cosine similarity between embeddings
print("\nComputing embedding similarity...")

# Sample 1000 embeddings for efficiency
sample_size = 1000
sample_indices = np.random.choice(mha_token_emb.shape[0], sample_size, replace=False)

mha_sample = mha_token_emb[sample_indices]
mqa_sample = mqa_token_emb[sample_indices]

# Compute pairwise cosine similarity
similarities = []
for i in range(sample_size):
    sim = cosine_similarity(mha_sample[i:i+1], mqa_sample[i:i+1])[0, 0]
    similarities.append(sim)

similarities = np.array(similarities)

print(f"\nEmbedding Similarity Statistics:")
print(f"  Mean cosine similarity: {similarities.mean():.4f}")
print(f"  Std cosine similarity: {similarities.std():.4f}")
print(f"  Min: {similarities.min():.4f}")
print(f"  Max: {similarities.max():.4f}")

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(similarities, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(similarities.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {similarities.mean():.3f}')
plt.xlabel('Cosine Similarity', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Token Embedding Similarity (MHA vs MQA)', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('embedding_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Similarity plot saved")

In [ ]:
# PCA comparison
print("\nApplying PCA to embeddings...")

# Fit PCA on MHA embeddings
pca = PCA(n_components=3, random_state=42)
mha_pca = pca.fit_transform(mha_token_emb)
mha_variance = pca.explained_variance_ratio_

# Transform MQA with same PCA
mqa_pca = pca.transform(mqa_token_emb)

# Also fit PCA separately on MQA to compare variance
pca_mqa = PCA(n_components=3, random_state=42)
mqa_pca_separate = pca_mqa.fit_transform(mqa_token_emb)
mqa_variance = pca_mqa.explained_variance_ratio_

print(f"\nPCA Explained Variance:")
print(f"\nMHA:")
for i, var in enumerate(mha_variance):
    print(f"  PC{i+1}: {var*100:.2f}%")
print(f"  Total: {mha_variance.sum()*100:.2f}%")

print(f"\nMQA:")
for i, var in enumerate(mqa_variance):
    print(f"  PC{i+1}: {var*100:.2f}%")
print(f"  Total: {mqa_variance.sum()*100:.2f}%")

print("\n✓ PCA complete")

In [ ]:
## 9. Trade-Off Analysis

## 8. Trade-Off Analysis

In [ ]:
# Create comprehensive comparison summary
print("\n" + "="*70)
print("COMPREHENSIVE TRADE-OFF ANALYSIS")
print("="*70)

# Determine winners
def get_winner(mha_val, mqa_val, lower_is_better=True):
    if lower_is_better:
        return 'MHA' if mha_val < mqa_val else 'MQA' if mqa_val < mha_val else 'Tie'
    else:
        return 'MHA' if mha_val > mqa_val else 'MQA' if mqa_val > mha_val else 'Tie'

tradeoff_data = {
    'Aspect': [
        'Parameter Count',
        'Model Size (MB)',
        'Validation Loss',
        'Validation Perplexity',
        'Generation Speed',
        'KV Cache Size',
        'Embedding Similarity',
        'PCA Variance Explained'
    ],
    'MHA': [
        f"{mha_total_params:,}",
        f"{mha_size_mb:.1f}",
        f"{mha_best_loss:.4f}" if isinstance(mha_best_loss, float) else 'N/A',
        f"{mha_val_ppl:.2f}" if isinstance(mha_val_ppl, (int, float)) else 'N/A',
        f"{mha_bench['tokens_per_sec']:.1f} tok/s",
        f"{mha_kv_cache / 1024:.1f} KB",
        'Baseline',
        f"{mha_variance.sum()*100:.1f}%"
    ],
    'MQA': [
        f"{mqa_total_params:,}",
        f"{mqa_size_mb:.1f}",
        f"{mqa_best_loss:.4f}" if isinstance(mqa_best_loss, float) else 'N/A',
        f"{mqa_val_ppl:.2f}" if isinstance(mqa_val_ppl, (int, float)) else 'N/A',
        f"{mqa_bench['tokens_per_sec']:.1f} tok/s",
        f"{mqa_kv_cache / 1024:.1f} KB",
        f"{similarities.mean():.3f}",
        f"{mqa_variance.sum()*100:.1f}%"
    ],
    'Winner': [
        get_winner(mha_total_params, mqa_total_params, lower_is_better=True),
        get_winner(mha_size_mb, mqa_size_mb, lower_is_better=True),
        get_winner(mha_best_loss, mqa_best_loss, lower_is_better=True) if isinstance(mha_best_loss, float) and isinstance(mqa_best_loss, float) else 'N/A',
        get_winner(mha_val_ppl, mqa_val_ppl, lower_is_better=True) if isinstance(mha_val_ppl, (int, float)) and isinstance(mqa_val_ppl, (int, float)) else 'N/A',
        get_winner(mha_bench['tokens_per_sec'], mqa_bench['tokens_per_sec'], lower_is_better=False),
        'MQA',
        'Similar',
        get_winner(mha_variance.sum(), mqa_variance.sum(), lower_is_better=False)
    ]
}

df_tradeoff = pd.DataFrame(tradeoff_data)
print("\n" + df_tradeoff.to_string(index=False))
print("\n" + "="*70)

In [ ]:
## 10. Conclusions & Recommendations

## 9. Conclusions & Recommendations

In [ ]:
## 11. Save Results

## 10. Save Results

In [ ]:
# Save comprehensive comparison report
report = {
    'models': {
        'mha': {
            'total_params': int(mha_total_params),
            'non_embed_params': int(mha_non_embed_params),
            'val_loss': float(mha_best_loss) if isinstance(mha_best_loss, float) else None,
            'val_ppl': float(mha_val_ppl) if isinstance(mha_val_ppl, (int, float)) else None,
            'tokens_per_sec': float(mha_bench['tokens_per_sec']),
            'checkpoint_size_mb': float(mha_size_mb)
        },
        'mqa': {
            'total_params': int(mqa_total_params),
            'non_embed_params': int(mqa_non_embed_params),
            'val_loss': float(mqa_best_loss) if isinstance(mqa_best_loss, float) else None,
            'val_ppl': float(mqa_val_ppl) if isinstance(mqa_val_ppl, (int, float)) else None,
            'tokens_per_sec': float(mqa_bench['tokens_per_sec']),
            'checkpoint_size_mb': float(mqa_size_mb)
        }
    },
    'differences': {
        'param_reduction_pct': float(total_diff),
        'non_embed_reduction_pct': float(non_embed_diff),
        'speedup': float(speedup),
        'kv_cache_reduction': float(mha_kv_cache / mqa_kv_cache)
    },
    'embeddings': {
        'cosine_similarity_mean': float(similarities.mean()),
        'cosine_similarity_std': float(similarities.std()),
        'mha_pca_variance': mha_variance.tolist(),
        'mqa_pca_variance': mqa_variance.tolist()
    },
    'generation_samples': generations
}

with open('model_comparison_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✓ Comparison report saved: model_comparison_report.json")

# List all generated files
print("\n📁 Generated Files:")
print("  • parameter_distribution.png")
print("  • training_metrics_comparison.png")
print("  • inference_performance.png")
print("  • embedding_similarity.png")
print("  • embedding_space_comparison.png")
print("  • speed_vs_quality_tradeoff.png")
print("  • model_comparison_report.json")